# Data Generator – Car Workshop & Accessories Shop Network

**Business scenario:** Network of 100 car workshops and accessories shops across Poland.  
**Period:** 2020-01 to 2026-12 (7 years)  
**Scale:** ~10-50 GB (configurable via SCALE_FACTOR)  

## Tables:
### Dimension
1. `dim_locations` - locations (100)
2. `dim_employees` - employees (~2000)
3. `dim_customers` - customers (~500K)
4. `dim_vehicles` - vehicles (~600K)
5. `dim_products` - products/parts (~15K)
6. `dim_services` - service catalogue (~200)
7. `dim_suppliers` - suppliers (~300)

### Fact
8. `fact_work_orders` - workshop work orders (~5M)
9. `fact_work_order_items` - work order items (~15M)
10. `fact_sales_transactions` - retail sales transactions (~30M)
11. `fact_sales_items` - sales items (~90M)
12. `fact_invoices` - invoices (~35M)
13. `fact_payments` - payments (~35M)
14. `fact_inventory_movements` - inventory movements (~50M)

### Supporting
15. `fact_appointments` - bookings (~5M)
16. `fact_purchase_orders` - supplier orders (~500K)
17. `fact_purchase_order_items` - purchase order items (~2M)
18. `fact_customer_feedback` - reviews (~2M)
19. `fact_loyalty_program` - loyalty programme (~500K)
20. `fact_employee_schedules` - work schedules (~3M)

In [0]:
import time

start = time.time()

In [0]:
# Instalacja zależności
!pip install pandas pyarrow faker tqdm

In [0]:
import os
import uuid
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timedelta, date
from faker import Faker
from tqdm import tqdm
import random
import uuid
import gc
import json
from pyspark.sql.types import *
from reference_data import *

fake = Faker('pl_PL')
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print('Biblioteki załadowane OK')

In [0]:
# ============================================================
# KONFIGURACJA
# ============================================================

# SCALE_FACTOR: 1.0 = pełne dane (~30GB), 0.1 = ~3GB, 0.01 = ~300MB do testów
SCALE_FACTOR = 0.1
#
# Katalog wyjściowy
FACT_OUTPUT_DIR = '/Volumes/car_workshop/fact/fact_files'
DIM_OUTPUT_DIR = '/Volumes/car_workshop/dim/dim_files'
# Format: 'parquet' lub 'csv'
OUTPUT_FORMAT = 'parquet'

# FACT_OUTPUT_DIR = './output/fact_daily_files'
# DIM_OUTPUT_DIR = './output/dim_daily_files'

# Okres danych
DATE_END = date.today()
DATE_START = DATE_END - timedelta(days=1)
NUM_YEARS = max(DATE_END.year - DATE_START.year + 1, 1)

# Rozmiar chunka przy zapisie (wiersze)
CHUNK_SIZE = 128_000

# Liczba lokalizacji
NUM_LOCATIONS = 100

os.makedirs(FACT_OUTPUT_DIR, exist_ok=True)
os.makedirs(DIM_OUTPUT_DIR, exist_ok=True)
print(f'SCALE_FACTOR = {SCALE_FACTOR}')
print(f'OUTPUT_FACT_DIR = {FACT_OUTPUT_DIR}')
print(f'OUTPUT_DIM_DIR = {DIM_OUTPUT_DIR}')
print(f'Szacowana wielkość danych: ~{SCALE_FACTOR * 30:.1f} GB')

In [0]:
# ============================================================
# HELPERY
# ============================================================

def save_table(df, table_name, partition_cols=None):
    """Zapisuje DataFrame jako parquet lub csv."""
    table_dir = os.path.join(DIM_OUTPUT_DIR, table_name)
    os.makedirs(table_dir, exist_ok=True)
    
    if OUTPUT_FORMAT == 'parquet':
        if partition_cols:
            pq.write_to_dataset(
                pa.Table.from_pandas(df),
                root_path=table_dir,
                partition_cols=partition_cols
            )
        else:
            pq.write_table(
                pa.Table.from_pandas(df),
                os.path.join(table_dir, f'{table_name}.parquet')
            )
    else:
        df.to_csv(os.path.join(table_dir, f'{table_name}.csv'), index=False)
    
    size_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f'  ✓ {table_name}: {len(df):,} wierszy, ~{size_mb:.1f} MB w pamięci')




def save_table_chunked(generate_func, table_name, total_rows, dir, partition_cols=None):
    """Generuje i zapisuje dane w chunkach aby oszczędzić RAM."""
    table_dir = os.path.join(dir, table_name)
    os.makedirs(table_dir, exist_ok=True)
    
    rows_written = 0
    chunk_num = 0
    
    with tqdm(total=total_rows, desc=table_name) as pbar:
        while rows_written < total_rows:
            chunk_rows = min(CHUNK_SIZE, total_rows - rows_written)
            df_chunk = generate_func(chunk_rows, rows_written)
            
            if OUTPUT_FORMAT == 'parquet':
                if partition_cols:
                    pq.write_to_dataset(
                        pa.Table.from_pandas(df_chunk),
                        root_path=table_dir,
                        partition_cols=partition_cols
                    )
                else:
                    chunk_id = uuid.uuid4().hex[:9]  # 9-znakowy hex, np. 'a3f9c1b27'
                    pq.write_table(
                        pa.Table.from_pandas(df_chunk),
                        os.path.join(table_dir, f'{table_name}_part_{chunk_id}.parquet')
                    )
            else:
                mode = 'w' if chunk_num == 0 else 'a'
                header = chunk_num == 0
                df_chunk.to_csv(
                    os.path.join(table_dir, f'{table_name}.csv'),
                    index=False, mode=mode, header=header
                )
            
            rows_written += chunk_rows
            chunk_num += 1
            pbar.update(chunk_rows)
            del df_chunk
            gc.collect()
    
    print(f'  ✓ {table_name}: {rows_written:,} wierszy w {chunk_num} chunkach')


def random_dates(start, end, n):
    """Generuje n losowych dat z zakresu z uwzględnieniem sezonowości."""
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    delta = (end_ts - start_ts).days
    random_days = np.random.randint(0, delta, size=n)
    dates = start_ts + pd.to_timedelta(random_days, unit='D')
    return dates


def seasonal_dates(start, end, n):
    """Generuje daty z sezonowością - więcej w okresach jesień/wiosna."""
    dates = random_dates(start, end, n)
    months = dates.month
    # Wagi sezonowe: więcej w marcu-kwietniu (wymiana opon) i październiku-listopadzie
    seasonal_weights = {1: 0.7, 2: 0.7, 3: 1.4, 4: 1.4, 5: 1.0, 6: 0.9,
                        7: 0.8, 8: 0.8, 9: 1.0, 10: 1.4, 11: 1.3, 12: 0.6}
    weights = np.array([seasonal_weights[m] for m in months])
    weights = weights / weights.sum()
    indices = np.random.choice(len(dates), size=n, replace=True, p=weights)
    return dates[indices]


def generate_uuid_batch(n):
    """Generuje batch UUID-ów."""
    return [str(uuid.uuid4()) for _ in range(n)]


print('Helpery załadowane OK')

In [0]:
# ============================================================
# TABLE SCHEMAS  (column -> Spark SQL type)
# Use these to enforce correct types when reading parquet files:
#   schema = StructType([StructField(c, t) for c, t in build_schema(TABLE_SCHEMAS['dim_locations'])])
# or pass directly to spark.read.schema(ddl_string).parquet(path)
# ============================================================

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType, DoubleType,
    BooleanType, DateType, TimestampType
)

_TYPE_MAP = {
    'STRING':    StringType(),
    'BIGINT':       IntegerType(),
    'BIGINT':    LongType(),
    'DOUBLE':    DoubleType(),
    'BOOLEAN':   BooleanType(),
    'DATE':      DateType(),
    'TIMESTAMP': TimestampType(),
}

def build_spark_schema(schema_dict):
    """Converts a schema dict {col: type_string} to a Spark StructType."""
    return StructType([
        StructField(col, _TYPE_MAP[typ], nullable=True)
        for col, typ in schema_dict.items()
    ])

def schema_to_ddl(schema_dict):
    """Converts a schema dict to a DDL string usable in spark.read.schema()."""
    return ', '.join(f'`{col}` {typ}' for col, typ in schema_dict.items())


# ------------------------------------------------------------------
# DIMENSION SCHEMAS
# ------------------------------------------------------------------

DIM_LOCATIONS_SCHEMA = {
    'location_id':     'BIGINT',
    'location_code':   'STRING',
    'name':            'STRING',
    'type':            'STRING',
    'street':          'STRING',
    'city':            'STRING',
    'region':          'STRING',
    'postal_code':     'STRING',
    'latitude':        'DOUBLE',
    'longitude':       'DOUBLE',
    'phone':           'STRING',
    'email':           'STRING',
    'manager_id':      'BIGINT',
    'number_of_bays':  'BIGINT',
    'area_m2':         'BIGINT',
    'opening_date':    'DATE',
    'is_active':       'BOOLEAN',
}

DIM_EMPLOYEES_SCHEMA = {
    'employee_id':        'BIGINT',
    'employee_code':      'STRING',
    'first_name':         'STRING',
    'last_name':          'STRING',
    'national_id':        'STRING',
    'position':           'STRING',
    'location_id':        'BIGINT',
    'hire_date':          'DATE',
    'termination_date':   'DATE',
    'hourly_rate':        'DOUBLE',
    'is_active':          'BOOLEAN',
}

DIM_CUSTOMERS_SCHEMA = {
    'customer_id':           'BIGINT',
    'customer_code':         'STRING',
    'customer_type':         'STRING',
    'first_name':            'STRING',
    'last_name':             'STRING',
    'company_name':          'STRING',
    'tax_id':                'STRING',
    'email':                 'STRING',
    'phone':                 'STRING',
    'city':                  'STRING',
    'postal_code':           'STRING',
    'registration_date':     'DATE',
    'preferred_location_id': 'BIGINT',
    'marketing_consent':     'BOOLEAN',
}

DIM_VEHICLES_SCHEMA = {
    'vehicle_id':              'BIGINT',
    'customer_id':             'BIGINT',
    'make':                    'STRING',
    'model':                   'STRING',
    'year':                    'BIGINT',
    'vin':                     'STRING',
    'registration_number':     'STRING',
    'fuel_type':               'STRING',
    'engine_displacement':     'DOUBLE',
    'horsepower':              'BIGINT',
    'color':                   'STRING',
    'mileage_km':              'BIGINT',
    'first_registration_date': 'DATE',
}

DIM_PRODUCTS_SCHEMA = {
    'product_id':          'BIGINT',
    'product_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'manufacturer':        'STRING',
    'purchase_price_net':  'DOUBLE',
    'sale_price_net':      'DOUBLE',
    'vat_rate':            'BIGINT',
    'unit':                'STRING',
    'weight_kg':           'DOUBLE',
    'min_stock_level':     'BIGINT',
    'is_active':           'BOOLEAN',
}

DIM_SERVICES_SCHEMA = {
    'service_id':          'BIGINT',
    'service_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'min_price_net':       'BIGINT',
    'max_price_net':       'BIGINT',
    'estimated_time_min':  'BIGINT',
    'is_active':           'BOOLEAN',
}

DIM_SUPPLIERS_SCHEMA = {
    'supplier_id':        'BIGINT',
    'supplier_code':      'STRING',
    'name':               'STRING',
    'tax_id':             'STRING',
    'city':               'STRING',
    'address':            'STRING',
    'postal_code':        'STRING',
    'phone':              'STRING',
    'email':              'STRING',
    'contact_person':     'STRING',
    'payment_terms_days': 'BIGINT',
    'min_order_value':    'DOUBLE',
    'is_active':          'BOOLEAN',
}

# ------------------------------------------------------------------
# FACT SCHEMAS
# ------------------------------------------------------------------

FACT_WORK_ORDERS_SCHEMA = {
    'work_order_id':       'BIGINT',
    'work_order_code':     'STRING',
    'location_id':         'BIGINT',
    'customer_id':         'BIGINT',
    'vehicle_id':          'BIGINT',
    'mechanic_id':         'BIGINT',
    'reception_date':      'DATE',
    'completion_date':     'DATE',
    'status':              'STRING',
    'mileage_at_reception':'BIGINT',
    'customer_notes':      'STRING',
    'year':                'INT',
    'month':               'INT',
}

FACT_WORK_ORDER_ITEMS_SCHEMA = {
    'wo_item_id':      'BIGINT',
    'work_order_id':   'BIGINT',
    'item_type':       'STRING',
    'service_id':      'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
    'discount_percent':'BIGINT',
}

FACT_SALES_TRANSACTIONS_SCHEMA = {
    'transaction_id':   'BIGINT',
    'transaction_code': 'STRING',
    'location_id':      'BIGINT',
    'customer_id':      'BIGINT',
    'employee_id':      'BIGINT',
    'transaction_date': 'DATE',
    'payment_method':   'STRING',
    'receipt_number':   'STRING',
    'year':             'INT',
    'month':            'INT',
}

FACT_SALES_ITEMS_SCHEMA = {
    'sales_item_id':   'BIGINT',
    'transaction_id':  'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'discount_percent':'BIGINT',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
}

FACT_INVOICES_SCHEMA = {
    'invoice_id':       'BIGINT',
    'invoice_code':     'STRING',
    'document_type':    'STRING',
    'source_type':      'STRING',
    'source_id':        'BIGINT',
    'customer_id':      'BIGINT',
    'location_id':      'BIGINT',
    'issue_date':       'DATE',
    'sale_date':        'DATE',
    'payment_due_date': 'DATE',
    'value_net':        'DOUBLE',
    'value_vat':        'DOUBLE',
    'value_gross':      'DOUBLE',
    'status':           'STRING',
    'year':             'INT',
    'month':            'INT',
}

FACT_PAYMENTS_SCHEMA = {
    'payment_id':         'BIGINT',
    'invoice_id':         'BIGINT',
    'payment_date':       'DATE',
    'amount':             'DOUBLE',
    'payment_method':     'STRING',
    'status':             'STRING',
    'transaction_number': 'STRING',
    'year':               'INT',
    'month':              'INT',
}

FACT_INVENTORY_MOVEMENTS_SCHEMA = {
    'movement_id':     'BIGINT',
    'product_id':      'BIGINT',
    'location_id':     'BIGINT',
    'movement_type':   'STRING',
    'quantity':        'BIGINT',
    'movement_date':   'DATE',
    'source_document': 'STRING',
    'document_number': 'STRING',
    'value_net':       'DOUBLE',
    'notes':           'STRING',
    'year':            'INT',
    'month':           'INT',
}

FACT_APPOINTMENTS_SCHEMA = {
    'appointment_id':   'BIGINT',
    'customer_id':      'BIGINT',
    'vehicle_id':       'BIGINT',
    'location_id':      'BIGINT',
    'service_id':       'BIGINT',
    'booking_date':     'DATE',
    'appointment_date': 'TIMESTAMP',
    'status':           'STRING',
    'booking_channel':  'STRING',
    'notes':            'STRING',
    'year':             'INT',
    'month':            'INT',
}

FACT_PURCHASE_ORDERS_SCHEMA = {
    'po_id':                  'BIGINT',
    'po_code':                'STRING',
    'supplier_id':            'BIGINT',
    'location_id':            'BIGINT',
    'order_date':             'DATE',
    'planned_delivery_date':  'DATE',
    'actual_delivery_date':   'DATE',
    'value_net':              'DOUBLE',
    'value_gross':            'DOUBLE',
    'status':                 'STRING',
    'year':                   'INT',
}

FACT_PURCHASE_ORDER_ITEMS_SCHEMA = {
    'po_item_id':         'BIGINT',
    'po_id':              'BIGINT',
    'product_id':         'BIGINT',
    'quantity_ordered':   'BIGINT',
    'quantity_delivered': 'BIGINT',
    'unit_price_net':     'DOUBLE',
    'value_net':          'DOUBLE',
}

FACT_CUSTOMER_FEEDBACK_SCHEMA = {
    'feedback_id':   'BIGINT',
    'customer_id':   'BIGINT',
    'location_id':   'BIGINT',
    'work_order_id': 'BIGINT',
    'feedback_date': 'DATE',
    'rating':        'BIGINT',
    'comment':       'STRING',
    'category':      'STRING',
    'channel':       'STRING',
}

FACT_LOYALTY_PROGRAM_SCHEMA = {
    'loyalty_id':    'BIGINT',
    'customer_id':   'BIGINT',
    'event_date':    'DATE',
    'event_type':    'STRING',
    'points':        'BIGINT',
    'description':   'STRING',
    'balance_after': 'BIGINT',
    'tier':          'STRING',
}

FACT_EMPLOYEE_SCHEDULES_SCHEMA = {
    'schedule_id':    'BIGINT',
    'employee_id':    'BIGINT',
    'date':           'DATE',
    'start_hour':     'BIGINT',
    'end_hour':       'BIGINT',
    'shift_type':     'STRING',
    'overtime_hours': 'BIGINT',
    'attendance':     'STRING',
}

# ------------------------------------------------------------------
# Master registry  {table_name: schema_dict}
# ------------------------------------------------------------------
TABLE_SCHEMAS = {
    'dim_locations':             DIM_LOCATIONS_SCHEMA,
    'dim_employees':             DIM_EMPLOYEES_SCHEMA,
    'dim_customers':             DIM_CUSTOMERS_SCHEMA,
    'dim_vehicles':              DIM_VEHICLES_SCHEMA,
    'dim_products':              DIM_PRODUCTS_SCHEMA,
    'dim_services':              DIM_SERVICES_SCHEMA,
    'dim_suppliers':             DIM_SUPPLIERS_SCHEMA,
    'fact_work_orders':          FACT_WORK_ORDERS_SCHEMA,
    'fact_work_order_items':     FACT_WORK_ORDER_ITEMS_SCHEMA,
    'fact_sales_transactions':   FACT_SALES_TRANSACTIONS_SCHEMA,
    'fact_sales_items':          FACT_SALES_ITEMS_SCHEMA,
    'fact_invoices':             FACT_INVOICES_SCHEMA,
    'fact_payments':             FACT_PAYMENTS_SCHEMA,
    'fact_inventory_movements':  FACT_INVENTORY_MOVEMENTS_SCHEMA,
    'fact_appointments':         FACT_APPOINTMENTS_SCHEMA,
    'fact_purchase_orders':      FACT_PURCHASE_ORDERS_SCHEMA,
    'fact_purchase_order_items': FACT_PURCHASE_ORDER_ITEMS_SCHEMA,
    'fact_customer_feedback':    FACT_CUSTOMER_FEEDBACK_SCHEMA,
    'fact_loyalty_program':      FACT_LOYALTY_PROGRAM_SCHEMA,
    'fact_employee_schedules':   FACT_EMPLOYEE_SCHEDULES_SCHEMA,
}

print(f'Table schemas loaded: {len(TABLE_SCHEMAS)} tables')
print()
print('Usage examples:')
print('  spark.read.schema(schema_to_ddl(TABLE_SCHEMAS["dim_locations"])).parquet(path)')
print('  spark.read.schema(build_spark_schema(TABLE_SCHEMAS["fact_work_orders"])).parquet(path)')

## 2. Tabele wymiarowe (dimension tables)

In [0]:
# ============================================================
# dim_locations - 100 lokalizacji warsztatów/sklepów
# ============================================================
print('Generowanie dim_locations...')

locations = []
for i, (city, region, lat, lon) in enumerate(CITIES[:NUM_LOCATIONS]):
    loc_type = np.random.choice(LOCATION_TYPES, p=LOCATION_TYPE_WEIGHTS)
    opening = fake.date_between(start_date=date(2005, 1, 1), end_date=date(2020, 6, 30))
    locations.append({
        'location_id': i + 1,
        'location_code': f'LOC-{i+1:03d}',
        'name': f'AutoService {city}',
        'type': loc_type,
        'street': fake.street_address(),
        'city': city,
        'region': region,
        'postal_code': fake.postcode(),
        'latitude': lat + np.random.uniform(-0.02, 0.02),
        'longitude': lon + np.random.uniform(-0.02, 0.02),
        'phone': fake.phone_number(),
        'email': f'service.{city.lower().replace(" ", "").replace("-", "")}@autoservice.pl',
        'manager_id': None,  # to be filled after generating employees
        'number_of_bays': np.random.randint(4, 12) if loc_type != 'shop' else 0,
        'area_m2': np.random.randint(200, 800),
        'opening_date': opening,
        'is_active': True if i < 95 else False,  # 5 locations closed
    })

DIM_LOCATIONS_SCHEMA = {
    'location_id':     'BIGINT',
    'location_code':   'STRING',
    'name':            'STRING',
    'type':            'STRING',
    'street':          'STRING',
    'city':            'STRING',
    'region':          'STRING',
    'postal_code':     'STRING',
    'latitude':        'DOUBLE',
    'longitude':       'DOUBLE',
    'phone':           'STRING',
    'email':           'STRING',
    'manager_id':      'BIGINT',
    'number_of_bays':  'BIGINT',
    'area_m2':         'BIGINT',
    'opening_date':    'DATE',
    'is_active':       'BOOLEAN',
}


# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_locations (
#   location_id              BIGINT,
#   location_code            STRING,
#   name                     STRING,
#   type                     STRING,
#   street                   STRING,
#   city                     STRING,
#   region                   STRING,
#   postal_code              STRING,
#   latitude                 DOUBLE,
#   longitude                DOUBLE,
#   phone                    STRING,
#   email                    STRING,
#   manager_id               BIGINT,
#   number_of_bays           BIGINT,
#   area_m2                  BIGINT,
#   opening_date             DATE,
#   is_active                BOOLEAN
# )
# USING DELTA;


df_locations_pd = pd.DataFrame(locations)
df_locations = spark.createDataFrame(df_locations_pd, schema=schema_to_ddl(DIM_LOCATIONS_SCHEMA))
save_table_chunked(lambda n, offset: df_locations.toPandas().iloc[offset:offset+n],  'dim_locations', len(df_locations_pd), DIM_OUTPUT_DIR, )
df_locations.head()

In [0]:
# ============================================================
# dim_employees - pracownicy (~20 na lokalizację = ~2000)
# ============================================================
print('Generating dim_employees...')

employees = []
emp_id = 1

for _, loc in df_locations_pd.iterrows():
    loc_id = loc['location_id']
    loc_type = loc['type']

    # Management staff - always present
    for position in POSITIONS['management']:
        employees.append({
            'employee_id': emp_id,
            'employee_code': f'EMP-{emp_id:05d}',
            'first_name': fake.first_name(),
            'last_name': fake.last_name(),
            'national_id': fake.pesel(),
            'position': position,
            'location_id': loc_id,
            'hire_date': fake.date_between(
                start_date=loc['opening_date'],
                end_date=min(loc['opening_date'] + timedelta(days=365), DATE_END)
            ),
            'termination_date': None,
            'hourly_rate': round(np.random.uniform(45, 80), 2),
            'is_active': loc['is_active'],
        })
        emp_id += 1

    # Workshop staff
    if loc_type in ('workshop', 'workshop_and_shop'):
        n_mechanics = np.random.randint(5, 10)
        for _ in range(n_mechanics):
            position = random.choice(POSITIONS['workshop'])
            employees.append({
                'employee_id': emp_id,
                'employee_code': f'EMP-{emp_id:05d}',
                'first_name': fake.first_name_male() if random.random() < 0.9 else fake.first_name_female(),
                'last_name': fake.last_name(),
                'national_id': fake.pesel(),
                'position': position,
                'location_id': loc_id,
                'hire_date': fake.date_between(
                    start_date=loc['opening_date'],
                    end_date=DATE_END
                ),
                'termination_date': fake.date_between(start_date=date(2022, 1, 1), end_date=DATE_END) if random.random() < 0.1 else None,
                'hourly_rate': round(np.random.uniform(30, 65), 2),
                'is_active': random.random() > 0.1,
            })
            emp_id += 1

    # Shop staff
    if loc_type in ('shop', 'workshop_and_shop'):
        n_sales = np.random.randint(3, 7)
        for _ in range(n_sales):
            position = random.choice(POSITIONS['shop'])
            employees.append({
                'employee_id': emp_id,
                'employee_code': f'EMP-{emp_id:05d}',
                'first_name': fake.first_name(),
                'last_name': fake.last_name(),
                'national_id': fake.pesel(),
                'position': position,
                'location_id': loc_id,
                'hire_date': fake.date_between(
                    start_date=loc['opening_date'],
                    end_date=DATE_END
                ),
                'termination_date': fake.date_between(start_date=date(2022, 1, 1), end_date=DATE_END) if random.random() < 0.15 else None,
                'hourly_rate': round(np.random.uniform(25, 45), 2),
                'is_active': random.random() > 0.12,
            })
            emp_id += 1

DIM_EMPLOYEES_SCHEMA = {
    'employee_id':        'BIGINT',
    'employee_code':      'STRING',
    'first_name':         'STRING',
    'last_name':          'STRING',
    'national_id':        'STRING',
    'position':           'STRING',
    'location_id':        'BIGINT',
    'hire_date':          'DATE',
    'termination_date':   'DATE',
    'hourly_rate':        'DOUBLE',
    'is_active':          'BOOLEAN',
}
# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_employees (
#   employee_id              BIGINT,
#   employee_code            STRING,
#   first_name               STRING,
#   last_name                STRING,
#   national_id              STRING,
#   position                 STRING,
#   location_id              BIGINT,
#   hire_date                DATE,
#   termination_date         DATE,
#   hourly_rate              DOUBLE,
#   is_active                BOOLEAN
# )
# USING DELTA;

df_employees_pd = pd.DataFrame(employees)
df_employees = spark.createDataFrame(df_employees_pd, schema=schema_to_ddl(DIM_EMPLOYEES_SCHEMA))
save_table_chunked(lambda n, offset: df_employees.toPandas().iloc[offset:offset+n],  'dim_locations', len(df_employees_pd), DIM_OUTPUT_DIR, )


# Lists of mechanic and seller IDs for use in fact tables
mechanic_ids = df_employees_pd[df_employees_pd['position'].isin(POSITIONS['workshop'])]['employee_id'].values
seller_ids = df_employees_pd[df_employees_pd['position'].isin(POSITIONS['shop'])]['employee_id'].values

# Mapping: location_id -> list of mechanics/sellers
loc_mechanics = df_employees_pd[df_employees_pd['position'].isin(POSITIONS['workshop'])].groupby('location_id')['employee_id'].apply(list).to_dict()
loc_sellers = df_employees_pd[df_employees_pd['position'].isin(POSITIONS['shop'])].groupby('location_id')['employee_id'].apply(list).to_dict()

print(f'  Mechanics: {len(mechanic_ids)}, Sales staff: {len(seller_ids)}')
df_employees.head()



In [0]:
# ============================================================
# dim_customers - klienci (500_000 * SCALE_FACTOR)
# ============================================================


NUM_CUSTOMERS = int(500_000 * SCALE_FACTOR)
print(f'Generating dim_customers ({NUM_CUSTOMERS:,} customers)...')

customer_types = np.random.choice(
    ['individual', 'business'], size=NUM_CUSTOMERS, p=[0.7, 0.3]
)

df_customers_pd = pd.DataFrame({
    'customer_id': np.arange(1, NUM_CUSTOMERS + 1),
    'customer_code': [f'CUS-{i:07d}' for i in range(1, NUM_CUSTOMERS + 1)],
    'customer_type': customer_types,
    'first_name': [fake.first_name() if t == 'individual' else '' for t in customer_types],
    'last_name': [fake.last_name() if t == 'individual' else '' for t in customer_types],
    'company_name': [fake.company() if t == 'business' else '' for t in customer_types],
    'tax_id': [fake.company_vat() if t == 'business' else '' for t in customer_types],
    'email': [fake.email() for _ in range(NUM_CUSTOMERS)],
    'phone': [fake.phone_number() for _ in range(NUM_CUSTOMERS)],
    'city': np.random.choice([m[0] for m in CITIES], size=NUM_CUSTOMERS),
    'postal_code': [fake.postcode() for _ in range(NUM_CUSTOMERS)],
    'registration_date': random_dates(DATE_START, DATE_END, NUM_CUSTOMERS),
    'preferred_location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=NUM_CUSTOMERS),
    'marketing_consent': np.random.choice([True, False], size=NUM_CUSTOMERS, p=[0.6, 0.4]),
})
 
DIM_CUSTOMERS_SCHEMA ={
  "customer_id"   :           "BIGINT",
  "customer_code" :           "STRING",
  "customer_type"  :          "STRING",
  "first_name"    :           "STRING",
  "last_name"     :           "STRING",
  "company_name"     :        "STRING",
  "tax_id":                   "STRING",
  "email"     :               "STRING",
  "phone"     :              "STRING",
  "city"   :                  "STRING",
  "postal_code"  :            "STRING",
  "registration_date":        "DATE",
  "preferred_location_id"  :  "BIGINT",
  "marketing_consent"   :     "BOOLEAN"

} 

df_customers = spark.createDataFrame(df_customers_pd, schema = schema_to_ddl(DIM_CUSTOMERS_SCHEMA))
# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_customers (
#   customer_id              BIGINT,
#   customer_code            STRING,
#   customer_type            STRING,
#   first_name               STRING,
#   last_name                STRING,
#   company_name             STRING,
#   tax_id                   STRING,
#   email                    STRING,
#   phone                    STRING,
#   city                     STRING,
#   postal_code              STRING,
#   registration_date        DATE,
#   preferred_location_id    BIGINT,
#   marketing_consent        BOOLEAN
# )
# USING DELTA;

# df_customers = customers

df_customers = spark.createDataFrame(df_customers_pd, schema=schema_to_ddl(DIM_CUSTOMERS_SCHEMA))
save_table_chunked(lambda n, offset: df_customers.toPandas().iloc[offset:offset+n],  'dim_locations', len(df_customers_pd), DIM_OUTPUT_DIR, )
customer_ids = df_customers_pd['customer_id'].values
df_customers.head()

In [0]:
# ============================================================
# dim_vehicles - pojazdy klientów (600K * SCALE_FACTOR)
# ============================================================
NUM_VEHICLES = int(600_000 * SCALE_FACTOR)
print(f'Generating dim_vehicles ({NUM_VEHICLES:,} vehicles)...')

makes = list(CAR_MAKES.keys())
weights = [MAKE_WEIGHTS[m] for m in makes]
weights_norm = np.array(weights) / sum(weights)

chosen_makes = np.random.choice(makes, size=NUM_VEHICLES, p=weights_norm)
chosen_models = [random.choice(CAR_MAKES[m]) for m in chosen_makes]

df_vehicles_pd = pd.DataFrame({
    'vehicle_id': np.arange(1, NUM_VEHICLES + 1),
    'customer_id': np.random.choice(customer_ids, size=NUM_VEHICLES),
    'make': chosen_makes,
    'model': chosen_models,
    'year': np.random.randint(2005, 2025, size=NUM_VEHICLES),
    'vin': [fake.bothify('???#########??????').upper() for _ in range(NUM_VEHICLES)],
    'registration_number': [fake.license_plate() for _ in range(NUM_VEHICLES)],
    'fuel_type': np.random.choice(FUEL_TYPES, size=NUM_VEHICLES, p=FUEL_WEIGHTS),
    'engine_displacement': np.random.choice(
        [1.0, 1.2, 1.4, 1.5, 1.6, 1.8, 2.0, 2.2, 2.5, 3.0],
        size=NUM_VEHICLES,
        p=[0.05, 0.10, 0.15, 0.12, 0.18, 0.12, 0.12, 0.06, 0.05, 0.05]
    ),
    'horsepower': np.random.randint(60, 350, size=NUM_VEHICLES),
    'color': np.random.choice(COLORS, size=NUM_VEHICLES),
    'mileage_km': np.random.randint(5000, 350000, size=NUM_VEHICLES),
    'first_registration_date': random_dates(date(2005, 1, 1), DATE_END, NUM_VEHICLES),
})

DIM_VEHICLES_SCHEMA = {
    'vehicle_id':              'BIGINT',
    'customer_id':             'BIGINT',
    'make':                    'STRING',
    'model':                   'STRING',
    'year':                    'BIGINT',
    'vin':                     'STRING',
    'registration_number':     'STRING',
    'fuel_type':               'STRING',
    'engine_displacement':     'DOUBLE',
    'horsepower':              'BIGINT',
    'color':                   'STRING',
    'mileage_km':              'BIGINT',
    'first_registration_date': 'DATE',
}

# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_vehicles (
#   vehicle_id                  BIGINT,
#   customer_id                 BIGINT,
#   make                        STRING,
#   model                       STRING,
#   year                        BIGINT,
#   vin                         STRING,
#   registration_number         STRING,
#   fuel_type                   STRING,
#   engine_displacement         DOUBLE,
#   horsepower                  BIGINT,
#   color                       STRING,
#   mileage_km                  BIGINT,
#   first_registration_date     date
# )
# USING DELTA;


# df_vehicles = vehicles
df_vehicles = spark.createDataFrame(df_vehicles_pd, schema=schema_to_ddl(DIM_VEHICLES_SCHEMA))
save_table_chunked(lambda n, offset: df_vehicles.toPandas().iloc[offset:offset+n],  'dim_vehicles', len(df_vehicles_pd), DIM_OUTPUT_DIR, )

# save_table_chunked(lambda n, offset: df_vehicles.iloc[offset:offset+n], 'dim_vehicles', len(df_vehicles), DIM_OUTPUT_DIR, )
vehicle_ids = df_vehicles_pd['vehicle_id'].values
print(f'  Average {NUM_VEHICLES / NUM_CUSTOMERS:.1f} vehicles per customer')
df_vehicles.head()

In [0]:
# ============================================================
# dim_products - produkty/części (~15K z wariantami)
# ============================================================
print('Generating dim_products...')

products = []
prod_id = 1

for category, product_list in PRODUCT_CATEGORIES.items():
    for base_name in product_list:
        # Each product has several variants (different manufacturers)
        manufacturers = random.sample(
            ['Bosch', 'Continental', 'Valeo', 'Hella', 'Mann', 'Mahle', 'NGK',
             'Brembo', 'TRW', 'KYB', 'Monroe', 'Sachs', 'LuK', 'Gates',
             'SKF', 'Dayco', 'Castrol', 'Mobil', 'Shell', 'Total', 'Motul',
             'Liqui Moly', 'K2', 'Meguiars', 'Sonax', 'Goodyear', 'Michelin',
             'Continental', 'Bridgestone', 'Pirelli', 'Varta', 'Exide', 'Banner'],
            k=min(random.randint(2, 6), 32)
        )
        for manufacturer in manufacturers:
            base_price = round(np.random.uniform(5, 800), 2)
            # Higher-price products: tyres, batteries, clutch
            if 'tyre' in base_name.lower():
                base_price = round(np.random.uniform(180, 600), 2)
            elif 'Battery' in base_name or 'AGM Battery' in base_name:
                base_price = round(np.random.uniform(250, 800), 2)
            elif 'Clutch kit' in base_name or 'Dual mass flywheel' in base_name:
                base_price = round(np.random.uniform(400, 2000), 2)
            elif 'shock absorber' in base_name.lower():
                base_price = round(np.random.uniform(100, 400), 2)
            elif 'filter' in base_name.lower():
                base_price = round(np.random.uniform(15, 80), 2)
            elif 'Brake pads' in base_name or 'Brake discs' in base_name:
                base_price = round(np.random.uniform(60, 300), 2)
            elif 'oil' in base_name.lower():
                base_price = round(np.random.uniform(30, 180), 2)
            elif 'bulb' in base_name.lower():
                base_price = round(np.random.uniform(8, 120), 2)

            margin = round(np.random.uniform(1.15, 1.45), 2)

            products.append({
                'product_id': prod_id,
                'product_code': f'PRD-{prod_id:06d}',
                'name': f'{base_name} {manufacturer}',
                'category': category,
                'manufacturer': manufacturer,
                'purchase_price_net': base_price,
                'sale_price_net': round(base_price * margin, 2),
                'vat_rate': 23,
                'unit': 'L' if category == 'Oils and Fluids' else 'pcs',
                'weight_kg': round(np.random.uniform(0.1, 15), 2),
                'min_stock_level': np.random.randint(2, 20),
                'is_active': random.random() > 0.05,
            })
            prod_id += 1
            
DIM_PRODUCTS_SCHEMA = {
    'product_id':          'BIGINT',
    'product_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'manufacturer':        'STRING',
    'purchase_price_net':  'DOUBLE',
    'sale_price_net':      'DOUBLE',
    'vat_rate':            'BIGINT',
    'unit':                'STRING',
    'weight_kg':           'DOUBLE',
    'min_stock_level':     'BIGINT',
    'is_active':           'BOOLEAN',
}

# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_products (
#   product_id               BIGINT,
#   product_code             STRING,
#   name                     STRING,
#   category                 STRING,
#   manufacturer             STRING,
#   purchase_price_net       DOUBLE,
#   sale_price_net           DOUBLE,
#   vat_rate                 BIGINT,
#   unit                     STRING,
#   weight_kg                DOUBLE,
#   min_stock_level          BIGINT,
#   is_active                BOOLEAN
# )
# USING DELTA;


df_products_pd = pd.DataFrame(products)
df_products = spark.createDataFrame(df_products_pd, schema=schema_to_ddl(DIM_PRODUCTS_SCHEMA))
save_table_chunked(lambda n, offset: df_products.toPandas().iloc[offset:offset+n],  'dim_products', len(df_products_pd), DIM_OUTPUT_DIR, )
product_ids = df_products_pd['product_id'].values
print(f'  Products: {len(df_products_pd):,} in {len(PRODUCT_CATEGORIES)} categories')
df_products.head()

In [0]:
# ============================================================
# dim_services - katalog usług warsztatowych
# ============================================================
print('Generating dim_services...')

services = []
for i, (name, category, min_p, max_p, duration) in enumerate(SERVICE_CATALOGUE):
    services.append({
        'service_id': i + 1,
        'service_code': f'SRV-{i+1:03d}',
        'name': name,
        'category': category,
        'min_price_net': min_p,
        'max_price_net': max_p,
        'estimated_time_min': duration,
        'is_active': True,
    })
    
DIM_SERVICES_SCHEMA = {
    'service_id':          'BIGINT',
    'service_code':        'STRING',
    'name':                'STRING',
    'category':            'STRING',
    'min_price_net':       'BIGINT',
    'max_price_net':       'BIGINT',
    'estimated_time_min':  'BIGINT',
    'is_active':           'BOOLEAN',
}

# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_services (
#   service_id               BIGINT,
#   service_code             STRING,
#   name                     STRING,
#   category                 STRING,
#   min_price_net            BIGINT,
#   max_price_net            BIGINT,
#   estimated_time_min       BIGINT,
#   is_active                BOOLEAN
# )
# USING DELTA;

df_services_pd = pd.DataFrame(services)
df_services = spark.createDataFrame(df_services_pd, schema=schema_to_ddl(DIM_SERVICES_SCHEMA))
save_table_chunked(lambda n, offset: df_services.toPandas().iloc[offset:offset+n],  'dim_services', len(df_services_pd), DIM_OUTPUT_DIR, )
service_ids = df_services['service_id'].values


# ============================================================
# dim_suppliers - dostawcy części
# ============================================================
DIM_SUPPLIERS_SCHEMA = {
    'supplier_id':        'BIGINT',
    'supplier_code':      'STRING',
    'name':               'STRING',
    'tax_id':             'STRING',
    'city':               'STRING',
    'address':            'STRING',
    'postal_code':        'STRING',
    'phone':              'STRING',
    'email':              'STRING',
    'contact_person':     'STRING',
    'payment_terms_days': 'BIGINT',
    'min_order_value':    'DOUBLE',
    'is_active':          'BOOLEAN',
}


# CREATE TABLE IF NOT EXISTS car_workshop.dim.dim_suppliers (
#   supplier_id              BIGINT,
#   supplier_code            STRING,
#   name                     STRING,
#   tax_id                   STRING,
#   city                     STRING,
#   address                  STRING,
#   postal_code              STRING,
#   phone                    STRING,
#   email                    STRING,
#   contact_person           STRING,
#   payment_terms_days       BIGINT,
#   min_order_value          DOUBLE,
#   is_active                BOOLEAN
# )
# USING DELTA;
print('Generating dim_suppliers...')

NUM_SUPPLIERS = 300
suppliers = []
for i in range(NUM_SUPPLIERS):
    suppliers.append({
        'supplier_id': i + 1,
        'supplier_code': f'SUP-{i+1:04d}',
        'name': fake.company(),
        'tax_id': fake.company_vat(),
        'city': random.choice([m[0] for m in CITIES]),
        'address': fake.street_address(),
        'postal_code': fake.postcode(),
        'phone': fake.phone_number(),
        'email': fake.company_email(),
        'contact_person': fake.name(),
        'payment_terms_days': random.choice([14, 21, 30, 45, 60]),
        'min_order_value': round(np.random.uniform(200, 2000), 2),
        'is_active': random.random() > 0.08,
    })


df_suppliers_pd = pd.DataFrame(suppliers)
df_suppliers = spark.createDataFrame(df_suppliers_pd, schema=schema_to_ddl(DIM_SUPPLIERS_SCHEMA))
save_table_chunked(lambda n, offset: df_suppliers.toPandas().iloc[offset:offset+n],  'dim_suppliers', len(df_suppliers_pd), DIM_OUTPUT_DIR, )
supplier_ids = df_suppliers_pd['supplier_id'].values

print(f'=== DIMENSION TABLES SUMMARY ===')
for name, df in [('dim_locations', df_locations_pd), ('dim_employees', df_employees_pd),
                  ('dim_customers', df_customers_pd), ('dim_vehicles', df_vehicles_pd),
                  ('dim_products', df_products_pd), ('dim_services', df_services_pd),
                  ('dim_suppliers', df_suppliers_pd)]:
    print(f'  {name}: {len(df):,} rows')

## 3. Tabele faktowe - zlecenia warsztatowe

In [0]:
# ============================================================
# fact_work_orders - zlecenia warsztatowe (5M * SCALE_FACTOR)
# ============================================================
NUM_WORK_ORDERS = int(700_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_work_orders ({NUM_WORK_ORDERS:,} work orders)...')

# Locations with workshop
workshop_locs = df_locations_pd[df_locations_pd['type'].isin(['workshop', 'workshop_and_shop'])]['location_id'].values

def generate_work_orders_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    loc_ids = np.random.choice(workshop_locs, size=chunk_size)

    # Assign mechanic from given location
    mech_ids = []
    for lid in loc_ids:
        mechs = loc_mechanics.get(lid, mechanic_ids[:5])
        mech_ids.append(random.choice(mechs))

    return pd.DataFrame({
        'work_order_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_code': [f'WO-{i:08d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'mechanic_id': mech_ids,
        'reception_date': dates,
        'completion_date': dates + pd.to_timedelta(np.random.randint(0, 5, size=chunk_size), unit='D'),
        'status': np.random.choice(WORK_ORDER_STATUSES, size=chunk_size, p=STATUS_WEIGHTS),
        'mileage_at_reception': np.random.randint(10000, 350000, size=chunk_size),
        'customer_notes': np.random.choice(
            ['', 'Knocking noise when braking', 'Engine losing power', 'Oil leak',
             'Seasonal tyre change', 'Periodic service', 'Air conditioning not cooling',
             'Engine warning light', 'Suspension noise', 'Brake pad replacement',
             'Preparation for inspection', 'Oil change', 'Starter motor problem',
             'Steering wheel vibration', 'Spark plug replacement', ''],
            size=chunk_size
        ),
        'year': dates.year,
        'month': dates.month,
    })

FACT_WORK_ORDERS_SCHEMA = {
    'work_order_id':       'BIGINT',
    'work_order_code':     'STRING',
    'location_id':         'BIGINT',
    'customer_id':         'BIGINT',
    'vehicle_id':          'BIGINT',
    'mechanic_id':         'BIGINT',
    'reception_date':      'DATE',
    'completion_date':     'DATE',
    'status':              'STRING',
    'mileage_at_reception':'BIGINT',
    'customer_notes':      'STRING',
    'year':                'INT',
    'month':               'INT',
}


# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_work_orders (
#   work_order_id              BIGINT,
#   work_order_code            STRING,
#   location_id                BIGINT,
#   customer_id                BIGINT,
#   vehicle_id                 BIGINT,
#   mechanic_id                BIGINT,
#   reception_date             date,
#   completion_date            date,
#   status                     STRING,
#   mileage_at_reception       BIGINT,
#   customer_notes             STRING,
#   year                       INT,
#   month                      INT
# )
# USING DELTA
# PARTITIONED BY (year, month);


fact_work_orders_pd = generate_work_orders_chunk(NUM_WORK_ORDERS, 0)
fact_work_orders = spark.createDataFrame(fact_work_orders_pd, schema=schema_to_ddl(FACT_WORK_ORDERS_SCHEMA))
save_table_chunked(lambda n, offset: fact_work_orders.toPandas().iloc[offset:offset+n],  'fact_work_orders', len(fact_work_orders_pd), FACT_OUTPUT_DIR, partition_cols=['year', 'month'])
fact_work_orders_ids = fact_work_orders_pd['work_order_id'].values

# save_table_chunked(generate_work_orders_chunk, 'fact_work_orders', NUM_WORK_ORDERS, FACT_OUTPUT_DIR,
#                    partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_work_order_items - pozycje zleceń (15M * SCALE_FACTOR)
# Każde zlecenie ma 1-6 pozycji (usługa + ewentualnie części)
# ============================================================
NUM_WO_ITEMS = int(15_000_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_work_order_items ({NUM_WO_ITEMS:,} items)...')

 
def generate_wo_items_chunk(chunk_size, offset):
    wo_ids = np.random.randint(1, NUM_WO_ITEMS + 1, size=chunk_size)
    # Random item type: service or part
    item_types = np.random.choice(['service', 'part'], size=chunk_size, p=[0.4, 0.6])

    srv_ids = np.where(
        item_types == 'service',
        np.random.choice(df_services_pd['service_id'].values, size=chunk_size),
        0
    )
    prod_ids = np.where(
        item_types == 'part',
        np.random.choice(product_ids, size=chunk_size),
        0
    )

    quantity = np.where(item_types == 'service', 1, np.random.randint(1, 5, size=chunk_size))
    unit_price = np.where(
        item_types == 'service',
        np.random.uniform(30, 2000, size=chunk_size),
        np.random.uniform(5, 500, size=chunk_size)
    )
    unit_price = np.round(unit_price, 2)

    return pd.DataFrame({
        'wo_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'work_order_id': wo_ids,
        'item_type': item_types,
        'service_id': srv_ids.astype(int),
        'product_id': prod_ids.astype(int),
        'quantity': quantity,
        'unit_price_net': unit_price,
        'value_net': np.round(unit_price * quantity, 2),
        'vat_rate': 23,
        'value_gross': np.round(unit_price * quantity * 1.23, 2),
        'discount_percent': np.random.choice([0, 0, 0, 5, 10, 15], size=chunk_size),
    })

FACT_WORK_ORDER_ITEMS_SCHEMA = {
    'wo_item_id':      'BIGINT',
    'work_order_id':   'BIGINT',
    'item_type':       'STRING',
    'service_id':      'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
    'discount_percent':'BIGINT',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_work_order_items (
#   wo_item_id                 BIGINT,
#   work_order_id              BIGINT,
#   item_type                  STRING,
#   service_id                 BIGINT,
#   product_id                 BIGINT,
#   quantity                   BIGINT,
#   unit_price_net             DOUBLE,
#   value_net                  DOUBLE,
#   vat_rate                   BIGINT,
#   value_gross                DOUBLE,
#   discount_percent           BIGINT
# )
# USING DELTA;


fact_work_orders_items_pd = generate_wo_items_chunk(NUM_WO_ITEMS, 0)
fact_work_orders_items = spark.createDataFrame(fact_work_orders_items_pd, schema=schema_to_ddl(FACT_WORK_ORDER_ITEMS_SCHEMA))
save_table_chunked(lambda n, offset: fact_work_orders_items_pd.iloc[offset:offset+n],  'fact_work_order_items', len(fact_work_orders_items_pd), FACT_OUTPUT_DIR)
fact_work_orders_ids = fact_work_orders_items_pd['wo_item_id'].values

## 4. Tabele faktowe - sprzedaż sklepowa

In [0]:
# ============================================================
# fact_sales_transactions - transakcje sprzedaży sklepowej (30M * SCALE_FACTOR)
# ============================================================
NUM_SALES = int(4_300_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_sales_transactions ({NUM_SALES:,} transactions)...')

# Locations with shop
shop_locs = df_locations_pd[df_locations_pd['type'].isin(['shop', 'workshop_and_shop'])]['location_id'].values

def generate_sales_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size) 
    hours = np.random.choice(range(7, 20), size=chunk_size,
                              p=[0.03, 0.08, 0.10, 0.10, 0.09, 0.08, 0.08,
                                 0.08, 0.08, 0.08, 0.08, 0.07, 0.05])
    minutes = np.random.randint(0, 60, size=chunk_size)

    timestamps = dates + pd.to_timedelta(hours, unit='h') + pd.to_timedelta(minutes, unit='m')
    loc_ids = np.random.choice(shop_locs, size=chunk_size)

    seller_arr = []
    for lid in loc_ids:
        sellers = loc_sellers.get(lid, seller_ids[:3])
        seller_arr.append(random.choice(sellers))

    # ~70% of transactions have a registered customer, ~30% are walk-in
    has_customer = np.random.random(size=chunk_size) < 0.7
    cust_ids = np.where(has_customer, np.random.choice(customer_ids, size=chunk_size), 0)

    return pd.DataFrame({
        'transaction_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_code': [f'TRX-{i:09d}' for i in range(offset + 1, offset + chunk_size + 1)],
        'location_id': loc_ids,
        'customer_id': cust_ids,
        'employee_id': seller_arr,
        'transaction_date': timestamps,
        'payment_method': np.random.choice(PAYMENT_METHODS, size=chunk_size, p=PAYMENT_WEIGHTS),
        'receipt_number': [f'REC/{random.randint(1,999):03d}/{i+offset+1:08d}' for i in range(chunk_size)],
        'year': dates.year,
        'month': dates.month,
    })
    
FACT_SALES_TRANSACTIONS_SCHEMA = {
    'transaction_id':   'BIGINT',
    'transaction_code': 'STRING',
    'location_id':      'BIGINT',
    'customer_id':      'BIGINT',
    'employee_id':      'BIGINT',
    'transaction_date': 'date',
    'payment_method':   'STRING',
    'receipt_number':   'STRING',
    'year':             'INT',
    'month':            'INT',
}


# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_sales_transactions (
#   transaction_id       BIGINT,
#   transaction_code     STRING,
#   location_id          BIGINT,
#   customer_id          BIGINT,
#   employee_id          BIGINT,
#   transaction_date     date,
#   payment_method       STRING,
#   receipt_number       STRING,
#   year                 INT,
#   month                INT
# )
# USING DELTA
# PARTITIONED BY (year, month);

fact_sales_transactions_pd = generate_sales_chunk(NUM_SALES, 0)
fact_sales_transactions = spark.createDataFrame(fact_sales_transactions_pd, schema=schema_to_ddl(FACT_SALES_TRANSACTIONS_SCHEMA))
save_table_chunked(lambda n, offset: fact_sales_transactions.toPandas().iloc[offset:offset+n],  'fact_sales_transactions', len(fact_sales_transactions_pd), FACT_OUTPUT_DIR)
fact_sales_transactions_ids = fact_sales_transactions_pd['transaction_id'].values

# save_table_chunked(generate_sales_chunk, 'fact_sales_transactions', NUM_SALES,  FACT_OUTPUT_DIR,
#                    partition_cols=['rok', 'miesiac'])

In [0]:
# ============================================================
# fact_sales_items - pozycje sprzedaży (90M * SCALE_FACTOR)
# Średnio 3 pozycje na transakcję
# ============================================================
NUM_SALES_ITEMS = int(24_000_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_sales_items ({NUM_SALES_ITEMS:,} items)...')

def generate_sales_items_chunk(chunk_size, offset):
    trx_ids = np.random.randint(1, NUM_SALES + 1, size=chunk_size)
    prod_ids_chunk = np.random.choice(product_ids, size=chunk_size)
    quantity = np.random.choice([1, 1, 1, 2, 2, 3, 4], size=chunk_size)
    unit_price = np.round(np.random.uniform(3, 600, size=chunk_size), 2)
    discount = np.random.choice([0, 0, 0, 0, 5, 10, 15, 20], size=chunk_size)
    value_after_discount = np.round(unit_price * quantity * (1 - discount / 100), 2)

    return pd.DataFrame({
        'sales_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'transaction_id': trx_ids,
        'product_id': prod_ids_chunk,
        'quantity': quantity,
        'unit_price_net': unit_price,
        'discount_percent': discount,
        'value_net': value_after_discount,
        'vat_rate': 23,
        'value_gross': np.round(value_after_discount * 1.23, 2),
    })

FACT_SALES_ITEMS_SCHEMA = {
    'sales_item_id':   'BIGINT',
    'transaction_id':  'BIGINT',
    'product_id':      'BIGINT',
    'quantity':        'BIGINT',
    'unit_price_net':  'DOUBLE',
    'discount_percent':'BIGINT',
    'value_net':       'DOUBLE',
    'vat_rate':        'BIGINT',
    'value_gross':     'DOUBLE',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_sales_items (
#   sales_item_id              BIGINT,
#   transaction_id             BIGINT,
#   product_id                 BIGINT,
#   quantity                   BIGINT,
#   unit_price_net             DOUBLE,
#   discount_percent           BIGINT,
#   value_net                  DOUBLE,
#   vat_rate                   BIGINT,
#   value_gross                DOUBLE
# )
# USING DELTA;

fact_sales_items_pd = generate_sales_items_chunk(NUM_SALES_ITEMS, 0)
fact_sales_items = spark.createDataFrame(fact_sales_items_pd, schema=schema_to_ddl(FACT_SALES_ITEMS_SCHEMA))
save_table_chunked(lambda n, offset: fact_sales_items.toPandas().iloc[offset:offset+n],  'fact_sales_items', len(fact_sales_items_pd), FACT_OUTPUT_DIR)
fact_sales_items_ids = fact_sales_items_pd['sales_item_id'].values


## 5. Tabele faktowe - faktury, płatności, magazyn

In [0]:
# ============================================================
# fact_invoices - faktury (35M * SCALE_FACTOR)
# Faktury powiązane z work_orders i sales_transactions
# ============================================================
NUM_INVOICES = int(5_000_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_invoices ({NUM_INVOICES:,} invoices)...')

def generate_invoices_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)

    # ~15% of invoices are for workshop orders, ~85% for retail sales
    source_type = np.random.choice(
        ['work_order', 'sales'], size=chunk_size, p=[0.15, 0.85]
    )
    source_ids = np.where(
        source_type == 'work_order',
        np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        np.random.randint(1, max(NUM_SALES, 1) + 1, size=chunk_size)
    )

    value_net = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    value_net = np.clip(value_net, 10, 50000)
    value_vat = np.round(value_net * 0.23, 2)

    # Type: VAT invoice, receipt, credit note
    document_type = np.random.choice(
        ['vat_invoice', 'receipt', 'credit_note'],
        size=chunk_size, p=[0.35, 0.60, 0.05]
    )

    return pd.DataFrame({
        'invoice_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_code': [f'INV/{dates[i].year}/{i+offset+1:08d}' for i in range(chunk_size)],
        'document_type': document_type,
        'source_type': source_type,
        'source_id': source_ids,
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'issue_date': dates,
        'sale_date': dates - pd.to_timedelta(np.random.randint(0, 3, size=chunk_size), unit='D'),
        'payment_due_date': dates + pd.to_timedelta(
            np.random.choice([0, 7, 14, 30], size=chunk_size, p=[0.5, 0.15, 0.2, 0.15]), unit='D'
        ),
        'value_net': value_net,
        'value_vat': value_vat,
        'value_gross': np.round(value_net + value_vat, 2),
        'status': np.random.choice(
            ['paid', 'pending', 'overdue', 'cancelled'],
            size=chunk_size, p=[0.80, 0.10, 0.07, 0.03]
        ),
        'year': dates.year,
        'month': dates.month,
    })

FACT_INVOICES_SCHEMA = {
    'invoice_id':       'BIGINT',
    'invoice_code':     'STRING',
    'document_type':    'STRING',
    'source_type':      'STRING',
    'source_id':        'BIGINT',
    'customer_id':      'BIGINT',
    'location_id':      'BIGINT',
    'issue_date':       'DATE',
    'sale_date':        'DATE',
    'payment_due_date': 'DATE',
    'value_net':        'DOUBLE',
    'value_vat':        'DOUBLE',
    'value_gross':      'DOUBLE',
    'status':           'STRING',
    'year':             'INT',
    'month':            'INT',
}


# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_invoices (
#   invoice_id          BIGINT,
#   invoice_code        STRING,
#   document_type       STRING,
#   source_type         STRING,
#   source_id           BIGINT,
#   customer_id         BIGINT,
#   location_id         BIGINT,
#   issue_date          date,
#   sale_date           date,
#   payment_due_date    date,
#   value_net           DOUBLE,
#   value_vat           DOUBLE,
#   value_gross         DOUBLE,
#   status              STRING,
#   year                INT,
#   month               INT
# )
# USING DELTA
# PARTITIONED BY (year, month);

fact_invoices_pd = generate_invoices_chunk(NUM_INVOICES, 0)
fact_invoices = spark.createDataFrame(fact_invoices_pd, schema=schema_to_ddl(FACT_INVOICES_SCHEMA))
save_table_chunked(lambda n, offset: fact_invoices.toPandas().iloc[offset:offset+n],  'fact_invoices', len(fact_invoices_pd), FACT_OUTPUT_DIR,  partition_cols=['year', 'month'])
fact_sales_items_ids = fact_invoices_pd['invoice_id'].values


In [0]:
# ============================================================
# fact_payments - płatności (35M * SCALE_FACTOR)
# ============================================================
NUM_PAYMENTS = int(5_000_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_payments ({NUM_PAYMENTS:,} payments)...')

def generate_payments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    amount = np.round(np.random.lognormal(mean=4.5, sigma=1.0, size=chunk_size), 2)
    amount = np.clip(amount, 5, 60000)

    return pd.DataFrame({
        'payment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'invoice_id': np.random.randint(1, max(NUM_INVOICES, 1) + 1, size=chunk_size),
        'payment_date': dates,
        'amount': amount,
        'payment_method': np.random.choice(PAYMENT_METHODS, size=chunk_size, p=PAYMENT_WEIGHTS),
        'status': np.random.choice(
            ['completed', 'pending', 'rejected', 'refund'],
            size=chunk_size, p=[0.90, 0.05, 0.03, 0.02]
        ),
        'transaction_number': [f'PAY-{uuid.uuid4().hex[:12].upper()}' for _ in range(chunk_size)],
        'year': dates.year,
        'month': dates.month,
    })

FACT_PAYMENTS_SCHEMA = {
    'payment_id':         'BIGINT',
    'invoice_id':         'BIGINT',
    'payment_date':       'DATE',
    'amount':             'DOUBLE',
    'payment_method':     'STRING',
    'status':             'STRING',
    'transaction_number': 'STRING',
    'year':               'INT',
    'month':              'INT',
}


# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_payments (
#   payment_id          BIGINT,
#   invoice_id          BIGINT,
#   payment_date        date,
#   amount              DOUBLE,
#   payment_method      STRING,
#   status              STRING,
#   transaction_number  STRING,
#   year                INT,
#   month               INT
# )
# USING DELTA
# PARTITIONED BY (year, month);

fact_payments_pd = generate_payments_chunk(NUM_PAYMENTS, 0)
fact_payments = spark.createDataFrame(fact_payments_pd, schema=schema_to_ddl(FACT_PAYMENTS_SCHEMA))
save_table_chunked(lambda n, offset: fact_payments.toPandas().iloc[offset:offset+n],  'fact_payments', len(fact_payments_pd), FACT_OUTPUT_DIR,  partition_cols=['year', 'month'])
fact_payments_ids = fact_payments_pd['payment_id'].values

In [0]:
# ============================================================
# fact_inventory_movements - ruchy magazynowe (50M * SCALE_FACTOR)
# ============================================================
NUM_INVENTORY = int(7_000_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_inventory_movements ({NUM_INVENTORY:,} movements)...')

def generate_inventory_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)

    movement_type = np.random.choice(
        ['receipt', 'issue_sales', 'issue_workshop', 'return', 'correction', 'stocktake'],
        size=chunk_size, p=[0.25, 0.35, 0.25, 0.05, 0.05, 0.05]
    )

    quantity = np.random.randint(1, 20, size=chunk_size)
    # Issues have negative quantity
    quantity = np.where(
        np.isin(movement_type, ['issue_sales', 'issue_workshop']),
        -quantity, quantity
    )

    return pd.DataFrame({
        'movement_id': np.arange(offset + 1, offset + chunk_size + 1),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'movement_type': movement_type,
        'quantity': quantity,
        'movement_date': dates,
        'source_document': np.random.choice(
            ['GR', 'GI', 'RM', 'RT', 'COR', 'INV'],
            size=chunk_size
        ),
        'document_number': [f'DOC-{i+offset+1:09d}' for i in range(chunk_size)],
        'value_net': np.round(np.abs(quantity) * np.random.uniform(5, 500, size=chunk_size), 2),
        'notes': np.random.choice(
            ['', '', '', 'Regular delivery', 'Special order',
             'Customer return', 'Stock correction', ''],
            size=chunk_size
        ),
        'year': dates.year,
        'month': dates.month,
    })


FACT_INVENTORY_MOVEMENTS_SCHEMA = {
    'movement_id':     'BIGINT',
    'product_id':      'BIGINT',
    'location_id':     'BIGINT',
    'movement_type':   'STRING',
    'quantity':        'BIGINT',
    'movement_date':   'DATE',
    'source_document': 'STRING',
    'document_number': 'STRING',
    'value_net':       'DOUBLE',
    'notes':           'STRING',
    'year':            'INT',
    'month':          'INT',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_inventory_movements (
#   movement_id        BIGINT,
#   product_id         BIGINT,
#   location_id        BIGINT,
#   movement_type      STRING,
#   quantity           BIGINT,
#   movement_date      date,
#   source_document    STRING,
#   document_number    STRING,
#   value_net          DOUBLE,
#   notes              STRING,
#   year               INT,
#   month              INT
# )
# USING DELTA
# PARTITIONED BY (year, month);

fact_inventory_movements_pd = generate_inventory_chunk(NUM_INVENTORY, 0)
fact_inventory_movements = spark.createDataFrame(fact_inventory_movements_pd, schema=schema_to_ddl(FACT_INVENTORY_MOVEMENTS_SCHEMA))
save_table_chunked(lambda n, offset: fact_inventory_movements.toPandas().iloc[offset:offset+n],  'fact_inventory_movements', len(fact_inventory_movements_pd), FACT_OUTPUT_DIR,  partition_cols=['year', 'month'])
fact_inventory_movements_ids = fact_inventory_movements_pd['movement_id'].values

## 6. Tabele wspierające

In [0]:
# ============================================================
# fact_appointments - rezerwacje wizyt (5M * SCALE_FACTOR)
# ============================================================
NUM_APPOINTMENTS = int(700_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_appointments ({NUM_APPOINTMENTS:,} bookings)...')

# Fix service_ids if it's a Column object
if hasattr(service_ids, '__class__') and 'Column' in str(type(service_ids)):
    service_ids_array = list(range(1, 201))  # Fallback to service ID range
else:
    service_ids_array = service_ids

def generate_appointments_chunk(chunk_size, offset):
    dates = seasonal_dates(DATE_START, DATE_END, chunk_size)
    hours = np.random.choice(range(7, 17), size=chunk_size)
    timestamps = dates + pd.to_timedelta(hours, unit='h')

    return pd.DataFrame({
        'appointment_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'vehicle_id': np.random.choice(vehicle_ids, size=chunk_size),
        'location_id': np.random.choice(workshop_locs, size=chunk_size),
        'service_id': np.random.choice(service_ids_array, size=chunk_size),
        'booking_date': dates - pd.to_timedelta(np.random.randint(1, 14, size=chunk_size), unit='D'),
        'appointment_date': timestamps,
        'status': np.random.choice(
            ['confirmed', 'completed', 'cancelled', 'no_show'],
            size=chunk_size, p=[0.10, 0.75, 0.10, 0.05]
        ),
        'booking_channel': np.random.choice(
            ['phone', 'online', 'in_person', 'email'],
            size=chunk_size, p=[0.35, 0.40, 0.15, 0.10]
        ),
        'notes': np.random.choice(
            ['', '', '', 'Please call before', 'Courtesy car needed',
             'Prefer morning', 'Urgent', 'Previously arranged', ''],
            size=chunk_size
        ),
        'year': dates.year,
        'month': dates.month,
    })
    
FACT_APPOINTMENTS_SCHEMA = {
    'appointment_id':   'BIGINT',
    'customer_id':      'BIGINT',
    'vehicle_id':       'BIGINT',
    'location_id':      'BIGINT',
    'service_id':       'BIGINT',
    'booking_date':     'DATE',
    'appointment_date': 'DATE',
    'status':           'STRING',
    'booking_channel':  'STRING',
    'notes':            'STRING',
    'year':             'INT',
    'month':            'INT',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_appointments (
#   appointment_id     BIGINT,
#   customer_id        BIGINT,
#   vehicle_id         BIGINT,
#   location_id        BIGINT,
#   service_id         BIGINT,
#   booking_date       date,
#   appointment_date   date,
#   status             STRING,
#   booking_channel    STRING,
#   notes              STRING,
#   year               INT,
#   month              INT
# )
# USING DELTA
# PARTITIONED BY (year, month);

fact_appointments_pd = generate_appointments_chunk(NUM_APPOINTMENTS, 0)
fact_appointments = spark.createDataFrame(fact_appointments_pd, schema=schema_to_ddl(FACT_APPOINTMENTS_SCHEMA))
save_table_chunked(lambda n, offset: fact_appointments.toPandas().iloc[offset:offset+n],  'fact_appointments', len(fact_appointments_pd), FACT_OUTPUT_DIR,  partition_cols=['year', 'month'])
fact_appointments_ids = fact_appointments_pd['appointment_id'].values

In [0]:
# ============================================================
# fact_purchase_orders - zamówienia do dostawców (500K * SCALE_FACTOR)
# ============================================================
NUM_PO = int(70_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_purchase_orders ({NUM_PO:,} orders)...')

def generate_po_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)
    value = np.round(np.random.lognormal(mean=7, sigma=0.8, size=chunk_size), 2)
    value = np.clip(value, 200, 100000)

    return pd.DataFrame({
        'po_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_code': [f'PO-{i+offset+1:07d}' for i in range(chunk_size)],
        'supplier_id': np.random.choice(supplier_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'order_date': dates,
        'planned_delivery_date': dates + pd.to_timedelta(np.random.randint(3, 21, size=chunk_size), unit='D'),
        'actual_delivery_date': dates + pd.to_timedelta(np.random.randint(3, 25, size=chunk_size), unit='D'),
        'value_net': value,
        'value_gross': np.round(value * 1.23, 2),
        'status': np.random.choice(
            ['placed', 'in_progress', 'delivered', 'partially_delivered', 'cancelled'],
            size=chunk_size, p=[0.03, 0.05, 0.85, 0.05, 0.02]
        ),
        'year': dates.year,
    })

FACT_PURCHASE_ORDERS_SCHEMA = {
    'po_id':                  'BIGINT',
    'po_code':                'STRING',
    'supplier_id':            'BIGINT',
    'location_id':            'BIGINT',
    'order_date':             'DATE',
    'planned_delivery_date':  'DATE',
    'actual_delivery_date':   'DATE',
    'value_net':              'DOUBLE',
    'value_gross':            'DOUBLE',
    'status':                 'STRING',
    'year':                   'INT',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_purchase_order_items (
#   po_id                        BIGINT,
#   po_code                      STRING,
#   supplier_id                  BIGINT,
#   location_id                  BIGINT,
#   order_date                   date,
#   planned_delivery_date        date,
#   actual_delivery_date         date,
#   value_net                    DOUBLE,
#   value_gross                  DOUBLE,
#   status                       STRING,
#   year                         INT
# )
# USING DELTA;


fact_purchase_orders_pd = generate_po_chunk(NUM_PO, 0)
fact_purchase_orders = spark.createDataFrame(fact_purchase_orders_pd, schema=schema_to_ddl(FACT_PURCHASE_ORDERS_SCHEMA))
save_table_chunked(lambda n, offset: fact_purchase_orders.toPandas().iloc[offset:offset+n],  'fact_purchase_orders', len(fact_purchase_orders_pd), FACT_OUTPUT_DIR)
fact_purchase_orders_ids = fact_purchase_orders_pd['po_id'].values

# ============================================================
# fact_purchase_order_items - pozycje zamówień (2M * SCALE_FACTOR)
# ============================================================
NUM_PO_ITEMS = int(2_000_000 * SCALE_FACTOR)
print(f'Generowanie fact_purchase_order_items ({NUM_PO_ITEMS:,} pozycji)...')

def generate_po_items_chunk(chunk_size, offset):
    ilosc = np.random.randint(1, 50, size=chunk_size)
    cena = np.round(np.random.uniform(5, 500, size=chunk_size), 2)
    
    return pd.DataFrame({
        'po_item_id': np.arange(offset + 1, offset + chunk_size + 1),
        'po_id': np.random.randint(1, max(NUM_PO, 1) + 1, size=chunk_size),
        'product_id': np.random.choice(product_ids, size=chunk_size),
        'ilosc_zamowiona': ilosc,
        'ilosc_dostarczona': np.clip(ilosc + np.random.randint(-2, 1, size=chunk_size), 0, 100),
        'cena_jednostkowa_netto': cena,
        'wartosc_netto': np.round(cena * ilosc, 2),
    })
FACT_PURCHASE_ORDER_ITEMS_SCHEMA = {
    'po_item_id':         'BIGINT',
    'po_id':              'BIGINT',
    'product_id':         'BIGINT',
    'quantity_ordered':   'BIGINT',
    'quantity_delivered': 'BIGINT',
    'unit_price_net':     'DOUBLE',
    'value_net':          'DOUBLE',
}


# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_purchase_order_items (
#   po_item_id                 BIGINT,
#   po_id                      BIGINT,
#   product_id                 BIGINT,
#   quantity_ordered           BIGINT,
#   quantity_delivered         BIGINT,
#   unit_price_net             DOUBLE,
#   value_net                  DOUBLE
# )
# USING DELTA;

fact_purchase_order_items_pd = generate_po_items_chunk(NUM_PO_ITEMS, 0)
fact_purchase_order_items = spark.createDataFrame(fact_purchase_order_items_pd, schema=schema_to_ddl(FACT_PURCHASE_ORDER_ITEMS_SCHEMA))
save_table_chunked(lambda n, offset: fact_purchase_order_items.toPandas().iloc[offset:offset+n],  'fact_purchase_order_items', len(fact_purchase_order_items_pd), FACT_OUTPUT_DIR)
fact_purchase_order_items_ids = fact_purchase_order_items_pd['po_item_id'].values

In [0]:
# ============================================================
# fact_customer_feedback - opinie klientów (2M * SCALE_FACTOR)
# ============================================================
NUM_FEEDBACK = int(285_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_customer_feedback ({NUM_FEEDBACK:,} reviews)...')

COMMENTS = [
    'Very professional service', 'Quick turnaround', 'Highly recommended!',
    'A bit overpriced', 'Long waiting time', 'Great communication',
    'Expert repair', 'Car ready ahead of schedule', 'Friendly staff',
    'Could be cheaper', 'Will definitely come back', 'Solid work',
    'Fair prices', 'Problem returned after a month', 'No complaints',
    'Excellent!', 'Average', 'Needs improvement', 'OK', '',
]

def generate_feedback_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)

    # Rating distribution: more positive ratings
    ratings = np.random.choice(
        [1, 2, 3, 4, 5], size=chunk_size,
        p=[0.03, 0.05, 0.12, 0.30, 0.50]
    )

    return pd.DataFrame({
        'feedback_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'location_id': np.random.randint(1, NUM_LOCATIONS + 1, size=chunk_size),
        'work_order_id': np.random.randint(1, max(NUM_WORK_ORDERS, 1) + 1, size=chunk_size),
        'feedback_date': dates,
        'rating': ratings,
        'comment': np.random.choice(COMMENTS, size=chunk_size),
        'category': np.random.choice(
            ['service_quality', 'repair_quality', 'turnaround_time', 'price', 'cleanliness', 'overall'],
            size=chunk_size, p=[0.20, 0.25, 0.15, 0.15, 0.10, 0.15]
        ),
        'channel': np.random.choice(
            ['google', 'online_form', 'email', 'phone'],
            size=chunk_size, p=[0.40, 0.30, 0.20, 0.10]
        ),
    })

FACT_CUSTOMER_FEEDBACK_SCHEMA = {
    'feedback_id':   'BIGINT',
    'customer_id':   'BIGINT',
    'location_id':   'BIGINT',
    'work_order_id': 'BIGINT',
    'feedback_date': 'DATE',
    'rating':        'BIGINT',
    'comment':       'STRING',
    'category':      'STRING',
    'channel':       'STRING',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_customer_feedback (
#   feedback_id      BIGINT,
#   customer_id      BIGINT,
#   location_id      BIGINT,
#   work_order_id    BIGINT,
#   feedback_date    date,
#   rating           BIGINT,
#   comment          STRING,
#   category         STRING,
#   channel          STRING
# )
# USING DELTA;


fact_customer_feedback_pd = generate_feedback_chunk(NUM_FEEDBACK, 0)
fact_customer_feedback = spark.createDataFrame(fact_customer_feedback_pd, schema=schema_to_ddl(FACT_CUSTOMER_FEEDBACK_SCHEMA))
save_table_chunked(lambda n, offset: fact_customer_feedback.toPandas().iloc[offset:offset+n],  'fact_customer_feedback', len(fact_customer_feedback_pd), FACT_OUTPUT_DIR)
fact_customer_feedback_ids = fact_customer_feedback_pd['feedback_id'].values

In [0]:
# ============================================================
# fact_loyalty_program - program lojalnościowy (500K * SCALE_FACTOR)
# ============================================================
NUM_LOYALTY = int(500_000 * SCALE_FACTOR)
print(f'Generating fact_loyalty_program ({NUM_LOYALTY:,} entries)...')

def generate_loyalty_chunk(chunk_size, offset):
    dates = random_dates(date(2021, 1, 1), DATE_END, chunk_size)  # Programme started in 2021

    return pd.DataFrame({
        'loyalty_id': np.arange(offset + 1, offset + chunk_size + 1),
        'customer_id': np.random.choice(customer_ids, size=chunk_size),
        'event_date': dates,
        'event_type': np.random.choice(
            ['points_earned', 'points_redeemed', 'bonus', 'expiry'],
            size=chunk_size, p=[0.60, 0.20, 0.10, 0.10]
        ),
        'points': np.random.choice(
            [-500, -200, -100, 10, 20, 50, 100, 200, 500],
            size=chunk_size, p=[0.05, 0.07, 0.08, 0.20, 0.25, 0.15, 0.10, 0.05, 0.05]
        ),
        'description': np.random.choice(
            ['Shop purchase', 'Workshop service', 'Welcome bonus',
             'Birthday bonus', 'Redeemed for 10% discount', 'Redeemed for 20% discount',
             'Redeemed for free service', 'Points expired', 'Referral bonus'],
            size=chunk_size
        ),
        'balance_after': np.random.randint(0, 5000, size=chunk_size),
        'tier': np.random.choice(
            ['standard', 'silver', 'gold', 'platinum'],
            size=chunk_size, p=[0.50, 0.30, 0.15, 0.05]
        ),
    })

FACT_LOYALTY_PROGRAM_SCHEMA = {
    'loyalty_id':    'BIGINT',
    'customer_id':   'BIGINT',
    'event_date':    'DATE',
    'event_type':    'STRING',
    'points':        'BIGINT',
    'description':   'STRING',
    'balance_after': 'BIGINT',
    'tier':          'STRING',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_loyalty_program (
#   loyalty_id       BIGINT,
#   customer_id      BIGINT,
#   event_date       date,
#   event_type       STRING,
#   points           BIGINT,
#   description      STRING,
#   balance_after    BIGINT,
#   tier             STRING
# )
# USING DELTA;

fact_loyalty_program_pd = generate_loyalty_chunk(NUM_LOYALTY, 0)
fact_loyalty_program = spark.createDataFrame(fact_loyalty_program_pd, schema=schema_to_ddl(FACT_LOYALTY_PROGRAM_SCHEMA))
save_table_chunked(lambda n, offset: fact_loyalty_program.toPandas().iloc[offset:offset+n],  'fact_loyalty_program', len(fact_loyalty_program_pd), FACT_OUTPUT_DIR)
fact_loyalty_program_ids = fact_loyalty_program_pd['loyalty_id'].values

In [0]:
# ============================================================
# fact_employee_schedules - grafiki pracy (3M * SCALE_FACTOR)
# ============================================================
NUM_SCHEDULES = int(430_000 * NUM_YEARS * SCALE_FACTOR)
print(f'Generating fact_employee_schedules ({NUM_SCHEDULES:,} entries)...')

all_employee_ids = df_employees.select('employee_id').toPandas()['employee_id'].values

def generate_schedules_chunk(chunk_size, offset):
    dates = random_dates(DATE_START, DATE_END, chunk_size)

    start_hour = np.random.choice([6, 7, 8, 9, 10, 12, 14], size=chunk_size,
                                   p=[0.05, 0.25, 0.30, 0.15, 0.05, 0.10, 0.10])
    work_hours = np.random.choice([4, 6, 8, 10, 12], size=chunk_size,
                                   p=[0.10, 0.10, 0.60, 0.15, 0.05])

    return pd.DataFrame({
        'schedule_id': np.arange(offset + 1, offset + chunk_size + 1),
        'employee_id': np.random.choice(all_employee_ids, size=chunk_size),
        'date': dates,
        'start_hour': start_hour,
        'end_hour': start_hour + work_hours,
        'shift_type': np.random.choice(
            ['day', 'morning', 'afternoon', 'night', 'day_off', 'holiday', 'sick_leave'],
            size=chunk_size, p=[0.40, 0.15, 0.15, 0.02, 0.15, 0.08, 0.05]
        ),
        'overtime_hours': np.random.choice(
            [0, 0, 0, 0, 0, 1, 2, 3, 4],
            size=chunk_size
        ),
        'attendance': np.random.choice(
            ['present', 'absent_excused', 'absent_unexcused', 'late'],
            size=chunk_size, p=[0.88, 0.08, 0.02, 0.02]
        ),
    })


FACT_EMPLOYEE_SCHEDULES_SCHEMA = {
    'schedule_id':    'BIGINT',
    'employee_id':    'BIGINT',
    'date':           'DATE',
    'start_hour':     'BIGINT',
    'end_hour':       'BIGINT',
    'shift_type':     'STRING',
    'overtime_hours': 'BIGINT',
    'attendance':     'STRING',
}

# CREATE TABLE IF NOT EXISTS car_workshop.fact.fact_employee_schedules (
#   schedule_id      BIGINT,
#   employee_id      BIGINT,
#   date             date,
#   start_hour       BIGINT,
#   end_hour         BIGINT,
#   shift_type       STRING,
#   overtime_hours   BIGINT,
#   attendance       STRING
# )
# USING DELTA;


fact_employee_schedules_pd = generate_schedules_chunk(NUM_SCHEDULES, 0)
fact_employee_schedules = spark.createDataFrame(fact_employee_schedules_pd, schema=schema_to_ddl(FACT_EMPLOYEE_SCHEDULES_SCHEMA))
save_table_chunked(lambda n, offset: fact_employee_schedules.toPandas().iloc[offset:offset+n],  'fact_employee_schedules', len(fact_employee_schedules_pd), FACT_OUTPUT_DIR)
fact_employee_schedules_ids = fact_employee_schedules_pd['schedule_id'].values

## 7. Walidacja i statystyki

In [0]:
# ============================================================
# VALIDATION AND SUMMARY
# ============================================================
import glob as glob_module

print('=' * 60)
print('DATA GENERATION SUMMARY')
print('=' * 60)
print(f'SCALE_FACTOR:       {SCALE_FACTOR}')
print(f'OUTPUT_DIR_DIM:     {DIM_OUTPUT_DIR}')
print(f'OUTPUT_DIR_FACT:    {FACT_OUTPUT_DIR}')
print()

total_size = 0
table_stats = []

# deduplicated in case both dirs are the same (local mode)
scan_dirs = list(dict.fromkeys([DIM_OUTPUT_DIR, FACT_OUTPUT_DIR]))

for base_dir in scan_dirs:
    if not os.path.isdir(base_dir):
        continue
    for table_name in sorted(os.listdir(base_dir)):
        table_path = os.path.join(base_dir, table_name)
        if os.path.isdir(table_path):
            # Count file sizes
            size = 0
            file_count = 0
            for root, dirs, files in os.walk(table_path):
                for f in files:
                    fp = os.path.join(root, f)
                    size += os.path.getsize(fp)
                    file_count += 1

            size_mb = size / (1024 * 1024)
            total_size += size
            table_stats.append({
                'table': table_name,
                'files': file_count,
                'size_MB': round(size_mb, 1),
            })

df_stats = pd.DataFrame(table_stats)
print(df_stats.to_string(index=False))
print()
print(f'TOTAL SIZE: {total_size / (1024**3):.2f} GB')
print(f'Estimated size at SCALE_FACTOR=1.0: ~{total_size / (1024**3) / SCALE_FACTOR:.1f} GB')
print()
print('Done! Output locations:')
print(f'  DIM:  {DIM_OUTPUT_DIR}')
print(f'  FACT: {FACT_OUTPUT_DIR}')
print()
print('To load data into Databricks:')
print('  1. Upload the output_data/ directory to DBFS or Unity Catalog Volume')
print('  2. Use spark.read.parquet("dbfs:/path/to/table/")')
print('  3. Or CREATE TABLE ... USING PARQUET LOCATION ...')

In [0]:
end = time.time()
print(f'Total time: {end - start:.2f} seconds')